# SVD & Low-Rank Approximation

Companion notebook for the [SVD & Low-Rank Approximation](https://ml-viz.vercel.app/courses/linear-algebra/04-svd-and-low-rank) lesson on ML Viz.

We'll verify the rotate–stretch–rotate picture, check Eckart–Young numerically, and compress an image with truncated SVD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

rng = np.random.default_rng(42)

## 1. SVD = rotate · stretch · rotate

U and V are orthogonal (rotations/reflections); Σ is diagonal (axis-aligned stretch). Reassembling them recovers A exactly.

In [ ]:
A = rng.normal(size=(5, 3))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print("U orthogonal:", np.allclose(U.T @ U, np.eye(3)))
print("V orthogonal:", np.allclose(Vt @ Vt.T, np.eye(3)))
print("singular values (sorted, non-negative):", s.round(3))
print("reconstruction exact:", np.allclose(U @ np.diag(s) @ Vt, A))

## 2. The PCA connection

Right singular vectors of centered data = principal components; σ²/n = explained variances.

In [ ]:
X = rng.normal(size=(500, 2)) @ np.array([[2.0, 1.2], [0.0, 0.6]])
Xc = X - X.mean(axis=0)

# PCA via covariance eigendecomposition
evals, evecs = np.linalg.eigh(Xc.T @ Xc / len(Xc))
# PCA via SVD
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)

print("eigen variances :", np.sort(evals)[::-1].round(4))
print("SVD variances   :", (s**2 / len(Xc)).round(4))

## 3. Eckart–Young, checked

The rank-k truncation error in Frobenius norm equals the root-sum-square of the discarded singular values — and no random rank-k matrix does better.

In [ ]:
A = rng.normal(size=(60, 40)) @ rng.normal(size=(40, 40))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

k = 10
A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
err = np.linalg.norm(A - A_k, "fro")
tail = np.sqrt((s[k:] ** 2).sum())
print(f"truncation error = {err:.4f}, discarded tail = {tail:.4f}")

# try to beat it with random rank-k matrices (spoiler: you can't)
best_random = min(
    np.linalg.norm(A - (rng.normal(size=(60, k)) @ np.linalg.lstsq(rng.normal(size=(60, k)), A, rcond=None)[0]), "fro")
    for _ in range(20)
)
print(f"best of 20 random rank-{k} attempts = {best_random:.4f}  (worse)")

## 4. Image compression

A synthetic 'image' compressed at increasing rank — watch structure return layer by layer.

In [ ]:
# build a structured image: gradients + blocks + a bit of noise
n = 128
yy, xx = np.mgrid[0:n, 0:n] / n
img = np.sin(6 * xx) + 0.8 * np.cos(4 * yy) + (xx > 0.5) * (yy > 0.5) * 1.5
img += 0.05 * rng.normal(size=img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, k in zip(axes, [1, 3, 10, 128]):
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    kept = 100 * (s[:k]**2).sum() / (s**2).sum()
    ax.imshow(approx, cmap="magma")
    ax.set_title(f"rank {k} · {kept:.1f}% energy", fontsize=10)
    ax.axis("off")
plt.suptitle("Truncated SVD: the best rank-k views of the image")
plt.show()

plt.figure(figsize=(7, 3))
plt.semilogy(s, color="#6366f1", lw=2)
plt.xlabel("index")
plt.ylabel("singular value (log)")
plt.title("Spectrum: a few large values, then a noise floor")
plt.grid(alpha=0.4)
plt.show()

**Next:** [PCA](https://ml-viz.vercel.app/courses/pca-dimensionality/01-pca) puts this machinery to work on data, and the [course quiz](https://ml-viz.vercel.app/courses/linear-algebra/05-quiz) checks the whole course.